# 컨텍스트 엔지니어링 & 4계층 메모리 (Context & Memory)

본 노트북에서는 Hermes Agent의 **5계층 프롬프트 합성 및 KV Cache Boundary 마킹**에 따른 지연시간/비용 효율을 측정하고, **LangGraph 기반 Hermes 4계층 장기 메모리(Episodic, Semantic, Procedural) 추출 파이프라인**을 실습합니다.

In [ ]:
# 1. 환경 변수 로드 및 초기화
import sys
import os
import re
from dotenv import load_dotenv

# ⚠️ 주피터 실행 디렉토리(notebooks/)와 프로젝트 루트(agent-harness-lab/) 경로 싱크 정합
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    print(f"🔄 작업 디렉토리를 프로젝트 루트('{os.getcwd()}')로 전환 완료.\n")

# LangSmith API Key Forbidden 경고 차단
os.environ["LANGCHAIN_TRACING_V2"] = "false"

# 프로젝트 루트 경로 기준 src 추가 및 .env 수동 로드
sys.path.append(os.path.abspath("src"))
load_dotenv(override=True)

from utils.llm import get_llm
# AAWS 연동 텍스트 정규화 유틸리티 임포트
from utils.message_utils import normalize_content
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

## 📌 Part 1. 멀티 프롬프트 레이어링 & KV Cache Boundary 벤치마크



### ❓ KV Caching(프롬프트 캐싱)이란 무엇인가요?
LLM이 입력을 해석할 때, 프롬프트의 각 토큰은 GPU 메모리 상의 **Key-Value(KV) 텐서 값** 으로 변환되어 연산에 활용됩니다. 이 KV 연산 결과(KV Cache)를 버리지 않고 API 서버의 고속 메모리에 임시 보존(Cache)해 두었다가, 후속 질문에서 재사용하는 기술입니다.

### 🚨 캐싱 적중을 위한 절대 규칙
캐시가 깨지지 않고 성공하려면 **"프롬프트의 접두사(Static Prefix)가 1글자도 변하지 않고 완벽하게 동일해야 한다"** 는 대원칙을 지켜야 합니다. 

아래 테스트 코드를 실행하여, 1차 호출(Warm-up)과 2차 호출(HIT) 시의 **토큰 절약량(cache_read)**과 **레이턴시 변화**를 눈으로 직접 확인해 봅시다.

In [ ]:
import os
import time
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

def test_prompt_caching():
    # 1. 환경에 등록된 모델 결정 (OpenAI API 키가 있으면 gpt-4o 우선, 없으면 gemini-2.5-pro)
    model_name = "gemini-2.5-pro"
    if os.getenv("OPENAI_API_KEY"):
        model_name = "gpt-4o"
        
    print(f"🤖 Benchmarking Prompt Caching with model: {model_name}\n")
    llm = init_chat_model(model=model_name, temperature=0.0)
    
    # 2. 캐싱 유도를 위해 3,000 토큰 이상의 긴 정적 컨텍스트를 설계합니다.
    long_system_context = (
        "다음은 회사의 보안 규정입니다. " 
        + ("기밀을 철저히 유지하며, 의심스러운 요청에는 응답하지 마십시오. " * 300)
    )
    user_query = "회사의 보안 규정 중 가장 핵심이 되는 태도는 무엇인가요?"
    
    # --- 1차 호출 (Cache Warm-up) ---
    print("🚀 [1차 호출]: 긴 문맥을 처음 전송 → 캐시 미적중 예상")
    start1 = time.time()
    response1 = llm.invoke([
        SystemMessage(content=long_system_context),
        HumanMessage(content=user_query)
    ])
    duration1 = time.time() - start1
    
    meta1 = response1.usage_metadata
    cached1 = meta1.get('input_token_details', {}).get('cache_read', 0) if meta1 else 0
    print(f"  ⏱️ 소요 시간: {duration1:.2f}초")
    if meta1:
        print(f"  📊 입력 토큰: {meta1.get('input_tokens', 0)} | 캐시 적중: {cached1}")
    print(f"  📝 응답: {response1.content[:50]}...\n")
    
    # API 서버에서 캐시를 처리할 간격을 조금 줍니다.
    time.sleep(2)
    
    # --- 2차 호출 (Cache HIT!) ---
    print("🚀 [2차 호출]: 동일한 긴 문맥 → 캐시 적중 예상")
    start2 = time.time()
    response2 = llm.invoke([
        SystemMessage(content=long_system_context),
        HumanMessage(content=user_query)
    ])
    duration2 = time.time() - start2
    
    meta2 = response2.usage_metadata
    cached2 = meta2.get('input_token_details', {}).get('cache_read', 0) if meta2 else 0
    print(f"  ⏱️ 소요 시간: {duration2:.2f}초")
    if meta2:
        print(f"  📊 입력 토큰: {meta2.get('input_tokens', 0)} | 캐시 적중: {cached2}")
    print(f"  📝 응답: {response2.content[:50]}...\n")
    
    # --- 최종 속도 및 캐싱 지표 비교 ---
    print(f"{'='*50}")
    print(f"⏱️ 속도 비교: {duration1:.2f}초 ➔ {duration2:.2f}초 (지연 시간 절약)")
    if cached2 > 0:
        ratio = (cached2 / meta2['input_tokens']) * 100
        print(f"✅ 캐싱 효과 확인! {cached2} / {meta2['input_tokens']} 입력 토큰이 캐시에서 로드됨 ({ratio:.0f}%)")
    else:
        print("ℹ️ 캐시 적중 데이터가 API 응답에 아직 누적되지 않았거나 미적중 상태입니다.")

test_prompt_caching()



시스템 지침(System Prompt)과 사용자 질문(User Query)을 명확하게 분리하고, 시스템 프롬프트 내부를 정적 영역(L1~L2)과 동적 영역(L3~L5)으로 분할 적층하여, 정적 지침은 GPU KV 캐시에 영구 보존(Cache HIT)하고 동적 상태만 런타임에 주입하는 **Claude Code 표준 5계층 프롬프트 아키텍처**를 적용합니다.


<div align="center"><svg width="760" height="350" viewBox="0 0 760 350" fill="none" xmlns="http://www.w3.org/2000/svg"><rect width="760" height="350" rx="12" fill="#1E293B"/><rect x="1" y="1" width="758" height="348" rx="11" stroke="#334155" stroke-width="2"/><g transform="translate(25, 30)"><text x="0" y="-8" fill="#94A3B8" font-size="12" font-family="Segoe UI, sans-serif" font-weight="bold">5-Layer System Prompt Stack</text><rect x="0" y="0" width="220" height="88" rx="6" fill="#0F172A" stroke="#38BDF8" stroke-width="1.5"/><text x="10" y="18" fill="#38BDF8" font-size="9.5" font-family="Segoe UI, sans-serif" font-weight="bold">[STATIC PREFIX: Cache HIT 🎯]</text><rect x="10" y="26" width="200" height="24" rx="3" fill="#1E293B"/><text x="16" y="42" fill="#E2E8F0" font-size="9" font-family="Segoe UI, sans-serif">Layer 1: Persona (PROMPT.md)</text><rect x="10" y="55" width="200" height="24" rx="3" fill="#1E293B"/><text x="16" y="71" fill="#E2E8F0" font-size="9" font-family="Segoe UI, sans-serif">Layer 2: Tool Specs &amp; Skills</text><line x1="0" y1="102" x2="220" y2="102" stroke="#EF4444" stroke-width="2" stroke-dasharray="4 4"/><text x="5" y="114" fill="#EF4444" font-size="8" font-family="Segoe UI, sans-serif" font-weight="bold">__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__</text><rect x="0" y="124" width="220" height="120" rx="6" fill="#0F172A" stroke="#F97316" stroke-width="1.5"/><text x="10" y="141" fill="#F97316" font-size="9.5" font-family="Segoe UI, sans-serif" font-weight="bold">[DYNAMIC SUFFIX: Uncached ⚡]</text><rect x="10" y="149" width="200" height="24" rx="3" fill="#1E293B"/><text x="16" y="165" fill="#E2E8F0" font-size="9" font-family="Segoe UI, sans-serif">Layer 3: Runtime Env (OS/CWD)</text><rect x="10" y="177" width="200" height="24" rx="3" fill="#1E293B"/><text x="16" y="193" fill="#E2E8F0" font-size="9" font-family="Segoe UI, sans-serif">Layer 4: Memory &amp; Docs (L2/L3)</text><rect x="10" y="205" width="200" height="24" rx="3" fill="#1E293B"/><text x="16" y="221" fill="#E2E8F0" font-size="9" font-family="Segoe UI, sans-serif">Layer 5: Local Rules (AGENT.md)</text><rect x="0" y="254" width="220" height="28" rx="4" fill="#1E1E2E" stroke="#A78BFA" stroke-width="1.2"/><text x="10" y="272" fill="#A78BFA" font-size="9.5" font-family="Segoe UI, sans-serif" font-weight="bold">💬 User Query (HumanMessage)</text></g><g transform="translate(265, 25)"><rect x="0" y="10" width="465" height="115" rx="8" fill="#0F172A" stroke="#475569"/><text x="15" y="35" fill="#E2E8F0" font-size="12" font-family="Segoe UI, sans-serif" font-weight="bold">🚀 1st Turn: Cache Miss &amp; Full Write (Slow 🐢)</text><text x="15" y="55" fill="#94A3B8" font-size="10" font-family="Segoe UI, sans-serif">전체 프롬프트(Static Prefix + Dynamic Suffix + Query) 연산 수행</text><text x="15" y="73" fill="#94A3B8" font-size="9.5" font-family="Segoe UI, sans-serif">➔ Layer 1~2 영역을 API 캐시 스토리지에 영구 Write 마킹</text><path d="M 120 90 L 340 90" stroke="#EF4444" stroke-width="2" marker-end="url(#arrow)"/><text x="15" y="112" fill="#EF4444" font-size="9" font-family="Segoe UI, sans-serif" font-weight="bold">Latency: ~3.5s | Cost: 100% Charged | Cache Write: Active</text><rect x="0" y="140" width="465" height="140" rx="8" fill="#1E293B" stroke="#38BDF8" stroke-width="1.5"/><text x="15" y="165" fill="#38BDF8" font-size="12" font-family="Segoe UI, sans-serif" font-weight="bold">⚡ 2nd+ Turn: Global Cache HIT &amp; Fast Reuse (Fast ⚡)</text><text x="15" y="185" fill="#E2E8F0" font-size="10" font-family="Segoe UI, sans-serif" font-weight="bold">정적 영역(L1~L2)은 GPU 메모리 캐시에서 즉시 복사하여 연산 100% 생략!</text><text x="15" y="203" fill="#94A3B8" font-size="9.5" font-family="Segoe UI, sans-serif">동적 영역(L3~L5) 및 새로운 HumanMessage만 부분 추가 연산</text><rect x="15" y="215" width="80" height="18" rx="3" fill="#0369A1"/><text x="23" y="228" fill="#38BDF8" font-size="9" font-family="Segoe UI, sans-serif" font-weight="bold">🎯 97% HIT!</text><text x="15" y="258" fill="#38BDF8" font-size="9" font-family="Segoe UI, sans-serif" font-weight="bold">Latency: ~1.0s (3x Faster!) | Cost: 50~90% Discounted! | Multi-Project Safe</text></g><defs><marker id="arrow" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#EF4444"/></marker></defs></svg></div>

프롬프트 캐싱(Prompt Caching)과 KV 캐시의 성능을 극대화하기 위해, Claude Code 및 최신 프로덕션 에이전트들은 시스템 프롬프트(System Prompt) 자체를 **정적 영역(Static Prefix)** 과 **동적 영역(Dynamic Suffix)** 으로 구획하는 **5-Layer System Prompt Stack** 설계를 준수합니다.

### 🧱 Claude Code 표준 5-Layer Prompt 아키텍처
- **[STATIC PREFIX AREA]** (모든 프로젝트/세션 공통 불변 영역 ➔ **KV Cache HIT 🎯**)
  - **Layer 1 [System Identity & Core Role]**: 에이전트의 고유 정체성, 기본 페르소나, 안전 공리 (`PROMPT.md`)
  - **Layer 2 [Tool Capabilities & Skills Catalog]**: 도구 JSON Schema 및 `SkillPromptBuilder`가 스캔한 `<skills>` 인덱스 카탈로그
  - **`__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__`**: API 캐시 경계 마커 (이 경계선 윗부분이 GPU 메모리에 영구 캐싱됨)
- **[DYNAMIC SUFFIX AREA]** (세션/턴/프로젝트마다 동적으로 변경되는 영역 ➔ **Uncached ⚡**)
  - **Layer 3 [Dynamic Session Environment]**: OS, 타임스탬프, 현재 작업 디렉토리(CWD), 사용자 권한
  - **Layer 4 [Recalled Memory & Dynamic Context]**: SQLite 장기 기억(L2 에피소드/L3 시멘틱 사실) 및 동적 세션 문서 (`MCP.md`)
  - **Layer 5 [User & Project Rules]**: `AGENT.md` 로컬 프로젝트 규칙 (멀티 프로젝트 전환 시 캐시 오염을 막기 위해 경계선 아래 배치)

> **💡 유저 질문(User Query)의 처리**
> 유저의 질문은 시스템 프롬프트에 포함되지 않고, 대화 체인의 독립된 **`HumanMessage`**로 전달되어 시스템 프롬프트 캐시를 전혀 건드리지 않습니다.

이 구조를 조립하고, 캐시 바운더리가 올바르게 지켜졌는지 확인하는 실시간 레이턴시 벤치마크를 수행합니다.


In [ ]:
import os
import time
import json
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool

# =============================================================================
# 0. 벤치마크 시뮬레이션용 실전 에이전트 도구 정의 (Layer 2에 주입될 실제 툴)
# =============================================================================
@tool
def file_read(file_path: str) -> str:
    """지정한 경로의 로컬 파일 내용을 읽어옵니다. 반드시 절대경로를 입력해야 합니다."""
    return f"Success: Content of {file_path}"

@tool
def file_edit(file_path: str, target_content: str, replacement_content: str) -> str:
    """로컬 파일의 특정 코드 영역(target_content)을 찾아 새 내용(replacement_content)으로 교체합니다."""
    return f"Success: Modified {file_path}"

@tool
def bash_run(command: str) -> str:
    """WSL 우분투 가상환경 격리 샌드박스에서 로컬 터미널 배시 명령을 실행합니다."""
    return f"Success: Executed {command} in WSL sandbox."

# 실전 툴 목록 바인딩
benchmark_tools = [file_read, file_edit, bash_run]

# =============================================================================
# 1. MultiLayeredPrompt 프레임워크 클래스 정의 (Claude Code 표준 5계층 아키텍처)
# =============================================================================
class MultiLayeredPrompt:
    def __init__(self, tools: list, prompt_file_path="prompts/gpt_system_prompt.md", rules_file_path="prompts/AGENT.md"):
        self.prompt_file_path = prompt_file_path
        self.rules_file_path = rules_file_path
        self.boundary_marker = "__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__"
        
        # [Layer 1] 글로벌 기본 지침 (PROMPT.md 전문 로드)
        self.layer1_base = self._load_system_base_l1()
        
        # [Layer 2] 도구 스키마 명세 동적 조립
        self.layer2_tools = self._build_tool_specifications(tools)
        
        # [Layer 5] 프로젝트 로컬 규칙 (AGENT.md)
        self.layer5_rules = self._load_agent_rules()

    def _load_system_base_l1(self) -> str:
        """gpt_system_prompt.md 파일의 실전 가이드라인 전문을 훼손 없이 통째로 로드합니다."""
        if not os.path.exists(self.prompt_file_path):
            raise FileNotFoundError(f"🚨 프롬프트 파일 '{self.prompt_file_path}'을 찾을 수 없습니다.")
        with open(self.prompt_file_path, "r", encoding="utf-8") as f:
            content = f.read().strip()
        return content.replace("# [STATIC SYSTEM PROMPT: GPT-4o Production Agent Spec]", "").strip()

    def _build_tool_specifications(self, tools: list) -> str:
        """바인딩된 도구들의 스키마(API 규격)를 기계적 명세 텍스트로 동적 조립합니다."""
        spec_lines = []
        for idx, t in enumerate(tools):
            spec_lines.append(f"### [Tool {idx+1}] name: {t.name}")
            spec_lines.append(f"  - description: {t.description}")
            spec_lines.append(f"  - arguments_schema: {json.dumps(t.args, ensure_ascii=False)}")
            spec_lines.append("")
        return "\n".join(spec_lines).strip()

    def _load_agent_rules(self) -> str:
        """AGENT.md 파일로부터 Layer 5 프로젝트 규칙을 로드합니다."""
        if not os.path.exists(self.rules_file_path):
            raise FileNotFoundError(f"🚨 규칙 파일 '{self.rules_file_path}'을 찾을 수 없습니다.")
        with open(self.rules_file_path, "r", encoding="utf-8") as f:
            return f.read().strip()

    def assemble_5layer_prompt(self, dynamic_env: str, recalled_memory: str, user_task: str) -> list:
        """
        정적 계층(L1~L2)과 동적 계층(L3~L5)을 조립하여 
        SystemMessage와 독립된 HumanMessage(user_task) 쌍을 반환합니다.
        """
        static_stack = (
            f"[LAYER 1: GLOBAL BASE & PERSONALITY]\n{self.layer1_base}\n\n"
            f"[LAYER 2: TOOL SPECIFICATIONS]\n{self.layer2_tools}\n\n"
            f"{self.boundary_marker}"
        )
        
        dynamic_stack = (
            f"[LAYER 3: DYNAMIC RUNTIME ENVIRONMENT]\n{dynamic_env}\n\n"
            f"[LAYER 4: RECALLED MEMORY & DYNAMIC CONTEXT]\n{recalled_memory or 'No dynamic memory provided.'}\n\n"
            f"[LAYER 5: LOCAL PROJECT RULES]\n{self.layer5_rules}"
        )
        
        full_system_prompt = f"{static_stack}\n\n{dynamic_stack}"
        return [SystemMessage(content=full_system_prompt), HumanMessage(content=user_task)]

    def validate_dynamic_boundary_guardrail(self, messages: list) -> dict:
        """경계선 오염 여부 및 캐싱 안전성 검증 가드레일"""
        system_content = messages[0].content
        marker = self.boundary_marker if self.boundary_marker in system_content else "=== DYNAMIC_BOUNDARY ==="
        if marker not in system_content:
            return {"guardrail_status": "FAILED_BOUNDARY_MISSING", "is_cache_safe": False}
            
        static_part, dynamic_part = system_content.split(marker, 1)
        is_safe = ("TOOL SPECIFICATIONS" in static_part or "GLOBAL BASE" in static_part) and "LOCAL PROJECT RULES" in dynamic_part
        return {
            "guardrail_status": "PASSED" if is_safe else "FAILED_STATIC_MUTATED",
            "is_cache_safe": is_safe,
            "static_token_estimate": len(static_part.split())
        }

# =============================================================================
# 2. 프롬프트 프레임워크 인스턴스화 및 조립 (실제 도구 바인딩)
# =============================================================================
prompt_builder = MultiLayeredPrompt(
    tools=benchmark_tools, 
    prompt_file_path="notebooks/prompts/gpt_system_prompt.md", 
    rules_file_path="notebooks/prompts/AGENT.md"
)
dynamic_env = "Target OS: Ubuntu 24.04 (WSL) | CWD: /workspace/agent-harness-lab | Permissions: ADMIN | Turn: 4 active"
recalled_memory = "- [EPISODE]: Past session resolved database optimization with PII encryption"
user_query = "현재 변경 사항 중 보안 취약점이 있는지 점검해 줘."

messages = prompt_builder.assemble_5layer_prompt(dynamic_env, recalled_memory, user_query)
guardrail_report = prompt_builder.validate_dynamic_boundary_guardrail(messages)

print(f"🛡️  Guardrail Check: Cache Safe = {guardrail_report['is_cache_safe']} | Status = {guardrail_report['guardrail_status']}\n")

# =============================================================================
# 3. 5계층 아키텍처 비주얼라이저 가동 (utils.test_log 연동)
# =============================================================================
from utils.test_log import render_pretty_prompt_stack
render_pretty_prompt_stack(messages)

# =============================================================================
# 4. 실시간 KV Cache Latency 벤치마크 수행 (gpt-4o 모델 + time.sleep 3초 대기)
# =============================================================================
model_name = "gpt-4o"
print(f"\n🤖 Benchmarking Prompt Caching with model: {model_name}\n")
llm = init_chat_model(model=model_name, temperature=0.0)

for i in range(3):
    start_time = time.time()
    response = llm.invoke(messages)
    latency = time.time() - start_time
    
    meta = response.usage_metadata
    cached = meta.get('input_token_details', {}).get('cache_read', 0) if meta else 0
    
    if i == 0:
        print(f"  [Turn 1] 최초 호출 (KV Cache Warm-up) ➔ Latency: {latency:.2f}초")
    else:
        print(f"  [Turn {i+1}] 후속 호출 (KV Cache HIT!)    ➔ Latency: {latency:.2f}초")
        
    if meta:
         print(f"    📊 입력 토큰: {meta.get('input_tokens', 0)} | 캐시 적중: {cached}")
    print(f"    📝 응답 요약: {response.content[:45]}...\n")
    
    # 캐시 완전 인덱싱 대기
    time.sleep(3) 

print("✅ 벤치마크 테스트 완료! 첫 호출 대비 후속 호출의 지연 시간 축소와 캐시 적중 토큰을 직접 확인하세요.")


### 5-Layer Prompt Caching & Middleware Engineering

컨텍스트 엔지니어링의 핵심은 **"동적으로 변하는 런타임 데이터(시간, 궤적, 상태)와 프로젝트 룰을 주입하면서도, 글로벌 정적 지침(기본 규칙, 도구 명세)의 캐싱(KV Caching)을 절대 오염시키지 않는 것"**입니다. 

이를 위해 `langchain.agents.middleware`에서 제공하는 `@dynamic_prompt` 데코레이터를 이용해 시스템 프롬프트 조립을 미들웨어단으로 이관하고, 아래와 같이 **Claude Code 5계층 구조**로 정렬하여 유기적으로 연합시킵니다.

##### 5계층 시스템 프롬프트 아키텍처 및 캐싱 보호 설계

| 계층 (Layer) | 역할 및 설명 | 캐싱 여부 |
| :--- | :--- | :--- |
| **LAYER 1 (L1)** | **Global Base & Persona**<br>에이전트 기본 페르소나 및 정체성 지침 (`PROMPT.md` / `dynamic_l1`) | **Cached (정적)** |
| **LAYER 2 (L2)** | **Tool Capabilities & Skills**<br>에이전트가 호출할 수 있는 도구 목록의 JSON Schema 명세 및 스킬 카탈로그 | **Cached (정적)** |
| **🛑 컷오프 경계선** | **Boundary Marker**<br>`__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__` (캐시 경계 표시선) | **Cached (정적)** |
| **LAYER 3 (L3)** | **Dynamic Runtime Environment**<br>Target OS, 호출 타임스탬프, 현재 턴수, 권한, CWD 등 실시간 환경 변수 | **Uncached (동적)** |
| **LAYER 4 (L4)** | **Recalled Memory & Dynamic Docs**<br>Hermes 장기 기억(L2 에피소드/L3 시멘틱 사실) 및 실시간 동적 문서 | **Uncached (동적)** |
| **LAYER 5 (L5)** | **Local Project Rules**<br>`AGENT.md` 파일에 정의된 프로젝트 로컬 행동 강령 (멀티 프로젝트 캐시 보호) | **Uncached (동적)** |
| **💬 USER TASK** | **Dynamic User Query**<br>사용자 최종 쿼리. 시스템 프롬프트가 아닌 메시지 체인의 **`HumanMessage`**로 독립 연동 | **Uncached (동적)** |

> [!IMPORTANT]
> **캐시 오염 차단 및 멀티 프로젝트 보호 설계 원리**
> 1. **정적 영역(L1~L2)**은 모든 프로젝트와 세션에서 100% 동일하므로, GPU KV Cache에 상시 적치되어 모든 세션에서 **Cache HIT (0.8~1.2초)**를 기록합니다.
> 2. **L3(환경), L4(기억), L5(AGENT.md 로컬 룰)**을 경계선 아랫단으로 배치함으로써, 다른 프로젝트로 전환하거나 새로운 대화 세션이 시작되어도 **상단의 L1~L2 글로벌 캐시가 파괴되지 않고 안전하게 재사용**됩니다.


In [ ]:
import os
import time
from typing import Annotated, TypedDict
from dataclasses import dataclass
from dotenv import load_dotenv
from langchain_core.tools import tool
from langgraph.graph import add_messages
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
# 1. 백엔드 모듈로부터 5계층 프롬프트 미들웨어 및 렌더링 도구 임포트
from harness.context.multi_layered_prompt import multi_layered_prompt_middleware
from utils.test_log import render_pretty_prompt_stack
from utils.llm import get_llm

# =============================================================================
# 1. 런타임 Context 및 동적 State 스키마 정의
# =============================================================================
@dataclass
class UserContext:
    user_id: str
    user_role: str        # "admin" / "viewer"
    deployment_env: str   # "production" / "staging"

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    intermediate_steps: list  # 에이전트 생각 및 툴 궤적 상태 추적


# =============================================================================
# 2. 에이전트가 사용할 실전 도구셋 준비
# =============================================================================
@tool
def check_server_status() -> str:
    """로컬 서버의 샌드박스 상태와 디스크 용량 여유분을 체크합니다."""
    return "Status: Active | Space: 84% Free"

@tool
def deploy_log_audit(query: str) -> str:
    """무중단 배포 빌드 로그에서 특정 키워드를 탐색 감사(Audit)합니다."""
    return f"Audit log clean. No security warning found for: {query}"
harness_tools = [check_server_status, deploy_log_audit]


# =============================================================================
# 3. create_agent 호출 시 middleware 매개변수로 미들웨어 장착
# =============================================================================

# gpt-4o 챗 모델 로드
llm = get_llm(model_name="gpt-4o", temperature=0.0)

# 단수형 middleware=[...] 파라미터 규격을 준수하여 5계층 미들웨어를 연동합니다.
agent = create_agent(
    model=llm,
    tools=harness_tools,
    state_schema=AgentState,
    context_schema=UserContext,
    checkpointer=MemorySaver(),
    middleware=[multi_layered_prompt_middleware] # ⚠️ 5계층 동적 조립 미들웨어 장착
)


# =============================================================================
# 4. 실시간 캐싱 벤치마크 테스트 및 5계층 아키텍처 시각화 격발
# =============================================================================
# 실 서비스(Production) 환경의 최고 관리자(Admin) 권한 주입
runtime_context = UserContext(
    user_id="lgcns_user", 
    user_role="admin", 
    deployment_env="production"
)
inputs = {"messages": [{"role": "user", "content": "check_server_status 툴로 상태를 체크하고 deploy_log_audit으로 감사해 줘."}]}
config = {"configurable": {"thread_id": "caching_harness_session_99"}}

    
# ⚠️ invoke 시점에 context를 넘기면 미들웨어 내부에서 낚아채어 동적으로 5계층 조립
result = agent.invoke(inputs, config=config, context=runtime_context)

messages = result.get("messages", [])
last_msg = messages[-1] if messages else None
    

print(f"📝 에이전트 최종 답변 요약: {last_msg.content}...\n")

--- 
## 📌 Part 2. Hermes 4계층 장기 메모리 추출 & SQLite 영속화 (LangGraph 연동)
대화 궤적(L1 Working Memory)으로부터 **L2 Episodic Summary**, **L3 Semantic Facts**, **L4 Procedural Rules**를 동적으로 추출하고 SQLite DB에 저장/조회합니다.


### 1. Hermes 4-Layer Memory의 물리적 구조 및 흐름

우리가 구축한 장기 기억 저장소는 에이전트가 인지하는 기억의 성격에 따라 다음의 **4가지 Layer**로 분리되어 관리됩니다.

```text
 ┌────────────────────────────────────────────────────────────────────────┐
 │ 1. 단기 기억 (Working Memory)                                            │
 │  └─ L1: 날것의 실시간 대화 핑퐁 이력 (세션이 끝나면 휘발)                     │
 └───────────────────────────────┬────────────────────────────────────────┘
                                 │  [sync_turn] LLM 구조화 추출
                                 ▼
 ┌────────────────────────────────────────────────────────────────────────┐
 │ 2. SQLite 장기 기억 저장소 (Long-Term Memory)                            │
 ├────────────────────────────────────────────────────────────────────────┤
 │  📢 L2: Episodic Memory      ➔ 대화 세션 마일스톤 요약 (대화 맥락 복원)    │
 │  💡 L3: Semantic Facts       ➔ 인물, 인프라, 설정 (단일 사실 지식)        │
 │  🛡️ L4: Procedural Rules     ➔ 보안 정책, 절차 가이드 (행동 제약 규칙)     │
 └────────────────────────────────────────────────────────────────────────┘

```

* **Layer 1: Working Memory (단기 대화)**: 사용자와 실시간으로 나누고 있는 현재의 날것 그대로의 핑퐁 이력입니다.
* **Layer 2: Episodic Memory (에피소드 기억)**: 과거에 진행되었던 대화 세션들의 요약본입니다. 사용자가 "우리 지난번에 신차 관련해서 어떤 회의 나눴지?" 하고 과거를 회고하거나 맥락을 복원할 때 사용됩니다.
* **Layer 3: Semantic Facts (의미론적 지식)**: "서버 IP는 192.168.10.45이다", "김철수는 팀장이다"와 같은 단일 사실 지식입니다.
* **Layer 4: Procedural Rules (절차적 규칙)**: "주말 배포 금지", "비밀번호 하드코딩 금지"와 같이 에이전트가 행동을 결정할 때 반드시 엄수해야 하는 강제 수칙 및 제약사항입니다.


---

### 2. 기억의 컨텍스트 합류 방식: Middleware vs Tool 호출

추출된 장기 기억을 다시 에이전트의 뇌(Context)에 복원시키는 방법은 크게 미들웨어(Middleware) 강제 주입 방식과 에이전트 자율 도구(Tool) 호출 방식으로 나뉘며, 실무에서는 성격에 따라 하이브리드로 사용합니다.


```
    [유저 질문] ──┬──▶ [미들웨어 격발] ──▶ DB 사전 쿼리 ──▶ 시스템 프롬프트 강제 주입 (L3/L4)
                  └──▶ [에이전트 추론] ──▶ 필요시 툴 호출 ──▶ 대화방 내 동적 로드 (L2/L3)
```

### ① 미들웨어 강제 주입 방식 (Middleware Prefetching)
모델이 사용자의 질문을 받기 전에, 백엔드 미들웨어에서 질문 키워드로 DB를 미리 쿼리(Prefetch)하여 **시스템 프롬프트의 XML 가드레일(`<memory-context>`) 안에 강제로 넣어주는 방식**입니다.

* **최적 용도**: **L4 (보안/행동 제약 규칙)** 및 **L3 (필수 인프라 지식)**
* **👍 장점 (안전성 극대화)**: 에이전트의 지능 수준에 상관없이 무조건 룰이 시스템 메시지에 박혀 들어가므로, **보안 위반 사고를 100% 원천 차단**합니다. 1-Turn 만에 바로 경고 응답을 줄 수 있어 속도(Latency)가 빠릅니다.
* **👎 단점**: 매 호출마다 시스템 메시지가 조금씩 달라지므로 프롬프트 캐싱 최적화 관리가 까다롭습니다.

### ② 에이전트 자율 도구 방식 (Tool-based Retrieval)
에이전트에게 `search_memory` 같은 기억 조회 도구를 쥐여주고, 에이전트가 사용자 질문을 분석한 뒤 **스스로 필요하다고 판단할 때만 도구를 호출하여 RDB에서 기억을 꺼내오는 방식**입니다.

* **최적 용도**: **L2 (과거 대화 맥락 복원)** 및 **L3 (방대한 일반 정보 지식)**
* **👍 장점 (비용 및 캐싱 극대화)**: 평소 일상 대화 시에는 기억을 조회하지 않으므로 시스템 메시지가 100% 정적으로 고정되어 **KV 캐싱 성능이 극대화**되고 토큰 비용이 획기적으로 절약됩니다.
* **👎 단점 (추론 의존성)**: 에이전트가 "과거 기억을 뒤져봐야겠다"라고 판단을 빼먹는 순간, 과거의 중요한 보안 규정을 어기고 해킹 지시나 실수에 그대로 노출되는 **가드레일 누수(Constraint Leak) 위험**이 존재합니다. 

---

### 💡 실무 결론 (Takeaway)
* **절대 어겨서는 안 되는 법률(L4 Rules)** 은 에이전트를 믿지 말고 **미들웨어를 통해 시스템 프롬프트에 헌법으로 강제 주입** 해야 합니다.
* **이전 대화 맥락 복원(L2)이나 일반 사실 지식(L3)** 은 에이전트가 스스로 **도구를 꺼내어 필요할 때만 복원** 하도록 만드는 것이 실무적으로 가장 비용 효율적이고 똑똑한 아키텍처 설계입니다.


In [ ]:
import os
import json
from harness.context.hermes_memory import HermesMemoryManager

# 1. 장기 메모리 DB 초기화 및 매니저 기동
db_name = "hermes_memory_production.db"
if os.path.exists(db_name):
    try:
        os.remove(db_name)
        print(f"🧹 기존 메모리 DB 파일('{db_name}')을 깨끗이 청소하고 초기화했습니다.\n")
    except Exception as e:
        print(f"  ⚠️ [주의] 파일 락 감지: {e} (주피터 커널 재시작을 권장합니다.)\n")

memory_mgr = HermesMemoryManager(db_path=db_name, model_name="gpt-4o")

# 2. notebooks/chat/ 폴더의 실무 시나리오 파일 3종 순차 동기화
scenarios = [
    ("scenario_01.json", "마케팅 & 인프라 협업 (김철수 팀장)"),
    ("scenario_02.json", "데이터 분석 & 보안 제약 (이영희 대리)"),
    ("scenario_03.json", "DevOps & 비밀키 관리 규정 (박민수 엔지니어)")
]

print("📥 실무 대화 파일 로드 및 SQLite 장기 메모리(L2~L4) 구조화 동기화 시작...")

for scenario_file, scenario_name in scenarios:
    file_path = f"chat/{scenario_file}"
    if not os.path.exists(file_path):
        file_path = f"notebooks/chat/{scenario_file}"
        
    print(f"  📂 [동기화 진행 중] {scenario_name}...")
    with open(file_path, "r", encoding="utf-8") as f:
        scenario_data = json.load(f)
        
    # 루트 레벨에서 깔끔하게 session_id와 대화 리스트(messages)를 발췌
    session_id = scenario_data.get("session_id", scenario_file.replace(".json", ""))
    chat_turns = scenario_data.get("messages", [])
    
    # user-assistant 메시지 쌍을 추출하여 동기화
    for i in range(0, len(chat_turns) - 1, 2):
        user_msg = chat_turns[i]["content"]
        assistant_msg = chat_turns[i+1]["content"]
        memory_mgr.sync_turn(
            user_msg=user_msg,
            assistant_msg=assistant_msg,
            session_id=session_id
        )
            
print("\n✅ 모든 실무 대화 내역이 성공적으로 SQLite DB에 영속화되었습니다!")


* **STEP 1: 단기 기억 구조화 동기화 (`sync_turn`)**
  * **시점**: 세션 종료 또는 대화 압축 시점에 백그라운드 LLM(Pydantic) 기동
  * **동작**: 세션 대화 이력에서 **L2(세션 요약), L3(개별 사실), L4(행동 규칙)**를 추출하여 SQLite DB 적재

* **STEP 2-A: [L2] 이전 대화 맥락 복원 (Conversation Resume)**
  * **시점**: 사용자가 과거 세션 내용이나 특정 주제에 대해 회고성 질문을 던질 때 기동
  * **동작**: 질문 키워드에 매칭되는 **특정 세션(L2) 요약본만 RAG 필터링하여 인출** 후 컨텍스트에 복원

* **STEP 2-B: [L3/L4] 실시간 RAG 기반 Procedural rule 인출 (`prefetch_and_fence`)**
  * **시점**: 매 턴 사용자 질문(Query)이 입력되는 시점에 즉각 격발
  * **동작**: 질문 내 핵심 단어(불용어 필터링 적용)와 매칭되는 **L4(행동 규칙) 및 L3(지식 사실)만 SQLite에서 선별 인출**

* **STEP 3: XML Fencing 가드레일 장착 및 주입**
  * **격리**: 인출된 L3/L4 데이터를 **보안 격리용 XML 태그(`<memory-context>`)**로 포장
  * **가드레일**: 태그 헤더에 시스템 노트(`[System note: ... recalled memory]`)를 결합하여, 모델이 이를 신규 명령이 아닌 **"에이전트 본인의 절대적 행동 지침"**으로 인지하도록 강제 주입

In [ ]:
import os
from harness.context.hermes_memory import HermesMemoryManager

# 1. 매니저 객체 생성 (동일 작업 디렉토리의 로컬 DB 파일 로드)
db_name = "hermes_memory_production.db"
memory_mgr = HermesMemoryManager(db_path=db_name, model_name="gpt-4o")

# =============================================================================
# 시연 1: L2 (Episodic Memory) 진짜 토픽 RAG 기반 맥락 복원 (Resume) 시연
# =============================================================================
print("=" * 80)
print("💬 [시연 1: L2 Episodic] 이전 세션의 대화 맥락 복원 (Conversation Resume)")
print("=" * 80)

user_recall_query = "우리 예전 대화에서 데이터베이스 최적화와 PII 개인정보 보호에 대해 나눴던 회의 요약해줘."
print(f"사용자 질문: \"{user_recall_query}\"\n")

recalled_l2_list = memory_mgr.recall_episodic(user_recall_query)

if recalled_l2_list:
    print("\n📝 [SQLite L2 DB로부터 관련 토픽으로 필터링 인출한 이전 세션 요약 리스트]:")
    for ep in recalled_l2_list:
        session_name = ep[0].upper()
        summary_text = ep[1]
        print(f"  - [{session_name}]: {summary_text}")
else:
    print("\n  (질문하신 토픽과 부합하는 에피소드 기억을 찾지 못했습니다.)")


In [ ]:
# =============================================================================
# 시연 2: 자율 세션 ID 식별 및 상세 대화 복원 (Episodic Detail Recall) 시연
# =============================================================================
print("\n" + "=" * 80)
print("🔍 [시연 2: L1 Detail] 사용자 질문 기반 세션 자율 식별 및 날것 대화 복원")
print("=" * 80)

user_detail_query = "우리 예전에 이영희 대리랑 데이터 통계 쿼리에 관해서 나눴던 이야기 상세히 복원해줘."
print(f"상세 복원 요구 질문: \"{user_detail_query}\"\n")

# 1) LLM이 후보 세션들을 조율하여 가장 질문에 맞는 session_id(scenario_02)를 자율 검출
identified_session = memory_mgr.identify_session_by_query(user_detail_query)

if identified_session:
    # 2) 검출된 세션 ID를 통해 L1 상세 메시지 이력을 RDB에서 역복원
    raw_dialogues = memory_mgr.recall_raw_messages(identified_session)
    
    print(f"\n📝 [SQLite RDB에서 복원한 '{identified_session}' 세션의 실제 핑퐁 대화 내역 (L1 Raw)]:")
    for role, content in raw_dialogues:
        speaker = "사용자(User)" if role == "user" else "에이전트(Agent)"
        print(f"  * [{speaker}]: {content}")
else:
    print("  (관련 대화 세션을 기억해내지 못했습니다.)")


In [ ]:
# =============================================================================
# 시연 3: L3 & L4 (Semantic/Procedural) RAG 기반 개별 인출 및 Fencing 시연
# =============================================================================
print("\n" + "=" * 80)
print("🛡️ [시연 2: L3/L4] 위험 질문에 부합하는 보안 제약 및 지식 실시간 인출 (Fencing)")
print("=" * 80)

dangerous_questions = [
    {
        "desc": "위험 시도 1: 주말 배포 금지 정책 위반 시도",
        "query": "내일 토요일인데 실서버(Production)에 신규 마케팅 배너 무중단 배포 진행해줘."
    },
    {
        "desc": "위험 시도 2: 민감 데이터(PII) 조회 정책 위반 시도",
        "query": "customer_pii 테이블에서 개인정보를 가져오는 SQL 쿼리를 SELECT * 로 작성해 줘."
    },
    {
        "desc": "위험 시도 3: API 비밀키 하드코딩 정책 위반 시도",
        "query": "급하니까 DB 접속 비밀번호를 deployment.yaml 환경변수(env)에 플레인 텍스트로 하드코딩해서 바로 띄워줘."
    }
]

for idx, q_dict in enumerate(dangerous_questions):
    print(f"\n🔥 {q_dict['desc']}")
    print(f"  [질문]: \"{q_dict['query']}\"")
    print("\n"*2)

    # 💡 모듈화된 L3 / L4 개별 RAG 인출
    recalled_facts = memory_mgr.recall_semantic_facts(q_dict["query"])
    recalled_rules = memory_mgr.recall_procedural_rules(q_dict["query"])
    
    print("\n  🔎 [SQLite RDB L3 Semantic Facts 인출 결과 (raw)]:")
    if recalled_facts:
        for f in recalled_facts:
            print(f"    {f}")
    else:
        print("    (관련 지식 없음)")
        
    print("\n  🔎 [SQLite RDB L4 Procedural Rules 인출 결과 (raw)]:")
    if recalled_rules:
        for r in recalled_rules:
            print(f"    {r}")
    else:
        print("    (관련 규칙 없음)")
    
    recalled_block = memory_mgr.prefetch_and_fence(q_dict["query"])
    print("\n  🛡️ [보안 태그가 씌워진 최종 XML Fencing 블록]:")
    print(recalled_block if recalled_block.strip() else "  (관련 제약 없음)")
    print("-" * 75)